# Debug: warum ist die Test-F1 (polygon-level) so viel schlechter als val_F1 (pixel-level)?

Training meldete `val_f1=0.7057` (pixelweise Maske, `train.py::compute_pixel_f1`).
`evaluate.py` (Instanz-Matching, IoU>=0.5) meldet auf dem Testsplit nur ~0.15 F1, mit sowohl schlechter Precision **als auch** schlechtem Recall.

Das Notebook geht der Reihe nach folgende Hypothesen durch:

1. **CRS/Geometrie-Mismatch** — `evaluate.py` liest die GT-Shapefiles ohne Reprojektion. **AUSGESCHLOSSEN**: CRS und Bounds von Raster und GT-Shapefile stimmen exakt ueberein (EPSG:25832).
2. **Ueber-Segmentierung durch watershed-Postprocessing** — pixelweise Maske ist ok, aber eine GT-Krone wird in mehrere kleine Pred-Polygone zerlegt. **BESTAETIGT per Sichtpruefung**: viele kleine rote Polygone liegen innerhalb einzelner gruener GT-Polygone. Das erklaert gleichzeitig niedrige Precision (Fragmente erreichen einzeln nie IoU>=0.5) und niedrigen Recall (keine GT-Krone wird von einem einzelnen Fragment ausreichend abgedeckt).
3. **Postprocessing-Schwellen unpassend** fuer dieses Modell/Datensatz (config nutzt baseline-Werte, nicht fuer dieses Modell getunt) — die Hebel gegen Ueber-Segmentierung sind vor allem `min_dist` und `sigma` (Watershed-Marker-Abstand/Glaettung), nicht `binary_threshold`/`label_threshold`.

Muss in der `nda`-Shell laufen (Datenzugriff), z.B. via `.venv/bin/jupyter lab --no-browser`.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("HSA_OVERRIDE_GFX_VERSION", "10.3.0")  # RX 6750 XT (gfx1031) -> gfx1030 kernels

import sys
from pathlib import Path

import numpy as np
import torch
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
import yaml

from deeptrees.model.deeptrees_model import DeepTreesModel
from deeptrees.modules.utils import predict_on_tile
from deeptrees.modules.postprocessing import extract_polygons

REPO = Path("/home/leafline/leafline")
sys.path.insert(0, str(REPO / "3_Model" / "src"))
from evaluate import match_polygons, compute_metrics, iou_polygon  # reuse exact scoring logic

with open(REPO / "3_Model/configs/finetune_v1.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base = Path(cfg["paths"]["base"])
stacked_dir = base / "stacked_6ch"
pp = cfg["postprocessing"]
print("device:", device)
print("postprocessing config:", pp)

## 1. Modell laden (best.pt)

In [ ]:
ckpt_path = REPO / "3_Model/runs/v1/checkpoints/best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("checkpoint epoch:", ckpt.get("epoch"), " best_f1 (pixel-level, val):", ckpt.get("best_f1"))

model = DeepTreesModel(
    in_channels=cfg["model"]["in_channels"],
    apply_sigmoid=True,
    lr=cfg["model"]["lr"],
    num_backbones=1,
)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
model = model.to(device)
print("model loaded")

## 2. Eine Testkachel waehlen und Inferenz laufen lassen

`AREA` wechseln, um `BotGarten` und `HoernNord` zu vergleichen.

In [ ]:
AREA = "BotGarten"  # oder "HoernNord"

item = next(i for i in cfg["data"]["test_areas"] if i["area"] == AREA)
stacked_path = stacked_dir / f"{AREA}.tif"
gt_shp = base / item["gt_file"]

with rasterio.open(stacked_path) as src:
    img = src.read().astype(np.float32)  # (6, H, W)
    transform = src.transform
    raster_crs = src.crs

tensor = torch.from_numpy(img).unsqueeze(0).to(device)

with torch.no_grad():
    output = predict_on_tile(
        model,
        tensor,
        patch_size=cfg["data"]["patch_size"],
        local_batch_size=cfg["training"]["batch_size"],
        stride=cfg["data"]["patch_size"] // 2,
    )

mask    = output[0, 0].cpu().numpy()
outline = output[0, 1].cpu().numpy()
dist    = output[0, 2].cpu().numpy()
print("raster shape:", img.shape, " raster CRS:", raster_crs)
print("mask/outline/dist range:", mask.min(), mask.max(), "|", outline.min(), outline.max(), "|", dist.min(), dist.max())

## 3. Hypothese 1: CRS-Mismatch zwischen Raster und GT-Shapefile — AUSGESCHLOSSEN

`evaluate.py` reprojiziert die GT-Geometrien **nicht** auf das Raster-CRS, bevor sie mit den predizierten Polygonen verglichen werden. `prepare_data.py::rasterize_gt` tut das beim Training sehr wohl (`gdf.to_crs(crs)`). Ergebnis: CRS und Bounds stimmen fuer BotGarten exakt ueberein (EPSG:25832) — kein Registrierungsfehler.

In [ ]:
gt_gdf_raw = gpd.read_file(gt_shp)
print("GT shapefile CRS:", gt_gdf_raw.crs)
print("Raster CRS       :", raster_crs)
print("MATCH:", gt_gdf_raw.crs == raster_crs)

print("\nGT bounds (raw): ", gt_gdf_raw.total_bounds)
print("Raster bounds   : ", rasterio.open(stacked_path).bounds)

# Reprojected version, used below for the rest of the notebook
gt_gdf = gt_gdf_raw.to_crs(raster_crs) if gt_gdf_raw.crs != raster_crs else gt_gdf_raw
print("\nGT bounds (reprojected to raster CRS):", gt_gdf.total_bounds)

## 4. Polygone extrahieren + Overlay-Visualisierung

Rot = Prediction, Gruen = Ground Truth. Bei korrekter Ausrichtung sollten sich die Konturen weitgehend decken.

In [ ]:
pred_polys = extract_polygons(mask, outline, dist, transform=transform, **pp)
gt_polys = list(gt_gdf.geometry)
print(f"pred={len(pred_polys)}  gt={len(gt_polys)}")

tp, fp, fn = match_polygons(pred_polys, gt_polys, threshold=0.5)
precision, recall, f1 = compute_metrics(tp, fp, fn)
print(f"tp={tp} fp={fp} fn={fn}  P={precision:.3f} R={recall:.3f} F1={f1:.3f}")

In [ ]:
def plot_overlay(ax, rgb, pred_polys, gt_polys, transform):
    ax.imshow(rgb)
    inv = ~transform  # world -> pixel

    def to_pixel_coords(poly):
        xs, ys = poly.exterior.coords.xy
        return [inv * (x, y) for x, y in zip(xs, ys)]

    for poly in gt_polys:
        coords = to_pixel_coords(poly)
        ax.add_patch(MplPolygon(coords, closed=True, fill=False, edgecolor="lime", linewidth=1.2))
    for poly in pred_polys:
        coords = to_pixel_coords(poly)
        ax.add_patch(MplPolygon(coords, closed=True, fill=False, edgecolor="red", linewidth=1.0))

rgb = np.clip(img[:3].transpose(1, 2, 0), 0, 1)

fig, ax = plt.subplots(figsize=(12, 12))
plot_overlay(ax, rgb, pred_polys, gt_polys, transform)
ax.set_title(f"{AREA} — rot=pred, gruen=GT")
plt.show()

In [ ]:
# Zoomed crop, damit einzelne Kronen erkennbar sind — Ausschnitt bei Bedarf anpassen
H, W = rgb.shape[:2]
r0, r1 = H // 3, H // 3 + 512
c0, c1 = W // 3, W // 3 + 512

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb[r0:r1, c0:c1])

inv = ~transform
def to_pixel_coords_cropped(poly):
    xs, ys = poly.exterior.coords.xy
    return [(px - c0, py - r0) for px, py in (inv * (x, y) for x, y in zip(xs, ys))]

for poly in gt_polys:
    coords = to_pixel_coords_cropped(poly)
    if any(0 <= x <= (c1 - c0) and 0 <= y <= (r1 - r0) for x, y in coords):
        ax.add_patch(MplPolygon(coords, closed=True, fill=False, edgecolor="lime", linewidth=1.5))
for poly in pred_polys:
    coords = to_pixel_coords_cropped(poly)
    if any(0 <= x <= (c1 - c0) and 0 <= y <= (r1 - r0) for x, y in coords):
        ax.add_patch(MplPolygon(coords, closed=True, fill=False, edgecolor="red", linewidth=1.2))

ax.set_title(f"{AREA} zoom — rot=pred, gruen=GT")
plt.show()

## 5. Rohe Wahrscheinlichkeitskarten (pixelweise) ansehen

Falls die Overlays komplett daneben liegen (Hypothese CRS), aber die Masken hier sinnvoll aussehen (Baumkronen-Form erkennbar), bestaetigt das: das Netz ist ok, das Problem liegt in Registrierung/Postprocessing, nicht im Modell selbst.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
axes[0].imshow(rgb); axes[0].set_title("RGB")
axes[1].imshow(mask, cmap="viridis", vmin=0, vmax=1); axes[1].set_title("mask (sigmoid)")
axes[2].imshow(outline, cmap="viridis", vmin=0, vmax=1); axes[2].set_title("outline (sigmoid)")
axes[3].imshow(dist, cmap="viridis"); axes[3].set_title("distance transform")
for a in axes:
    a.axis("off")
plt.show()

## 6. IoU-Verteilung: Fehlausrichtung vs. schwache Erkennung vs. Ueber-/Untersegmentierung

Fuer jedes Pred-Polygon die beste IoU zu irgendeinem GT-Polygon (und umgekehrt). Interpretation:

- **fast alle IoUs ~0** -> Registrierungsfehler (CRS/Transform), keine echte inhaltliche Ueberlappung.
- **viele IoUs zwischen 0.1-0.4** -> Modell/Postprocessing findet Baeume ungefaehr an der richtigen Stelle, aber Kronenform/Groesse stimmt nicht (Ueber-/Untersegmentierung, watershed-Parameter) -> IoU-0.5-Schwelle ist evtl. zu hart dafuer.
- **bimodale Verteilung (viele ~0, einige hoch)** -> Modell trifft manche Baeume gut, andere komplett verpasst -> Recall-Problem bei bestimmten Baumtypen/Groessen.

In [ ]:
def best_ious(polys_a, polys_b):
    out = []
    for a in polys_a:
        best = 0.0
        for b in polys_b:
            best = max(best, iou_polygon(a, b))
        out.append(best)
    return np.array(out)

pred_best_iou = best_ious(pred_polys, gt_polys)
gt_best_iou = best_ious(gt_polys, pred_polys)

print(f"pred polys with best IoU == 0 : {(pred_best_iou == 0).sum()} / {len(pred_best_iou)}")
print(f"gt polys with best IoU == 0   : {(gt_best_iou == 0).sum()} / {len(gt_best_iou)}")
print(f"pred median best IoU (>0 only): {np.median(pred_best_iou[pred_best_iou > 0]) if (pred_best_iou > 0).any() else float('nan'):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(pred_best_iou, bins=20); axes[0].set_title("best IoU pro Pred-Polygon")
axes[1].hist(gt_best_iou, bins=20); axes[1].set_title("best IoU pro GT-Polygon")
plt.show()

## 7. Fragmentierungs-Check: wie viele Pred-Polygone pro GT-Krone?

Direktes Mass fuer Ueber-Segmentierung: fuer jede GT-Krone, wie viele Pred-Polygone ueberlappen sie ueberhaupt (nicht nur IoU>=0.5, sondern jede nennenswerte Ueberschneidung)? Ein hoher Mittelwert (>1) bestaetigt quantitativ, was im Overlay-Plot visuell auffiel.

In [ ]:
import geopandas as gpd

pred_sindex_gdf = gpd.GeoDataFrame(geometry=pred_polys)
sindex = pred_sindex_gdf.sindex

frag_counts = []
for g in gt_polys:
    candidate_idx = list(sindex.intersection(g.bounds))
    n_overlapping = sum(
        1 for i in candidate_idx
        if pred_polys[i].intersection(g).area > 0.1 * pred_polys[i].area
    )
    frag_counts.append(n_overlapping)

frag_counts = np.array(frag_counts)
print(f"GT-Kronen ohne ueberlappendes Pred-Polygon : {(frag_counts == 0).sum()} / {len(frag_counts)}")
print(f"GT-Kronen mit genau 1 Pred-Polygon         : {(frag_counts == 1).sum()} / {len(frag_counts)}")
print(f"GT-Kronen mit >1 Pred-Polygonen (fragmentiert): {(frag_counts > 1).sum()} / {len(frag_counts)}")
print(f"Mittlere Fragment-Anzahl (nur getroffene GT) : {frag_counts[frag_counts > 0].mean() if (frag_counts > 0).any() else float('nan'):.2f}")

plt.figure(figsize=(6, 4))
plt.hist(frag_counts, bins=range(0, frag_counts.max() + 2))
plt.xlabel("Anzahl ueberlappender Pred-Polygone pro GT-Krone")
plt.title(f"{AREA} — Fragmentierung")
plt.show()

## 8. Postprocessing-Sweep gegen Ueber-Segmentierung: `min_dist` und `sigma`

Das sind die Watershed-Hebel gegen zu viele Marker pro Krone (siehe `find_treecrowns_from_dist_trafo` in `deeptrees/modules/postprocessing.py`):
- `min_dist`: Mindestabstand zwischen lokalen Maxima (`corner_peaks(..., min_distance=min_dist)`) — groesser = weniger, groessere Marker pro Krone.
- `sigma`: Gaussian-Glaettung vor der Maxima-Suche — groesser = spurious lokale Maxima werden weggeglaettet.

`binary_threshold`/`label_threshold` bleiben hier auf Config-Werten fixiert, weil sie primaer steuern *ob* ueberhaupt etwas erkannt wird, nicht *wie viele Marker* pro Objekt.

In [ ]:
import itertools

results = []
for md, sg in itertools.product([10, 15, 20, 25, 30, 40], [1, 2, 3, 4]):
    pp_try = {**pp, "min_dist": md, "sigma": sg}
    polys = extract_polygons(mask, outline, dist, transform=transform, **pp_try)
    tp_, fp_, fn_ = match_polygons(polys, gt_polys, threshold=0.5)
    p_, r_, f1_ = compute_metrics(tp_, fp_, fn_)
    results.append((md, sg, len(polys), tp_, fp_, fn_, p_, r_, f1_))

print(f"{'min_dist':>8} {'sigma':>5} {'npred':>6} {'tp':>4} {'fp':>4} {'fn':>4} {'P':>6} {'R':>6} {'F1':>6}")
for md, sg, npred, tp_, fp_, fn_, p_, r_, f1_ in sorted(results, key=lambda r: -r[-1])[:15]:
    print(f"{md:8d} {sg:5d} {npred:6d} {tp_:4d} {fp_:4d} {fn_:4d} {p_:6.3f} {r_:6.3f} {f1_:6.3f}")
print(f"\n(zum Vergleich, gt={len(gt_polys)} — je naeher npred an gt, desto weniger Fragmentierung)")

## 9. Postprocessing-Sweep: `binary_threshold`/`label_threshold`

Zusaetzlich zur Fragmentierung: testet, ob die Erkennungs-Schwellen aus der config (baseline-Werte) fuer dieses Modell unpassend sind. Mit den Werten aus Abschnitt 8 kombinieren, sobald dort ein gutes `min_dist`/`sigma`-Paar gefunden ist.

In [ ]:
results = []
for bt, lt in itertools.product([0.05, 0.10, 0.20, 0.30], [0.05, 0.10, 0.20, 0.30]):
    pp_try = {**pp, "binary_threshold": bt, "label_threshold": lt}
    polys = extract_polygons(mask, outline, dist, transform=transform, **pp_try)
    tp_, fp_, fn_ = match_polygons(polys, gt_polys, threshold=0.5)
    p_, r_, f1_ = compute_metrics(tp_, fp_, fn_)
    results.append((bt, lt, len(polys), tp_, fp_, fn_, p_, r_, f1_))

print(f"{'bt':>5} {'lt':>5} {'npred':>6} {'tp':>4} {'fp':>4} {'fn':>4} {'P':>6} {'R':>6} {'F1':>6}")
for bt, lt, npred, tp_, fp_, fn_, p_, r_, f1_ in sorted(results, key=lambda r: -r[-1]):
    print(f"{bt:5.2f} {lt:5.2f} {npred:6d} {tp_:4d} {fp_:4d} {fn_:4d} {p_:6.3f} {r_:6.3f} {f1_:6.3f}")

## Fazit-Checkliste

- [x] CRS von GT-Shapefile == Raster-CRS? (Abschnitt 3) — **ja, ausgeschlossen als Ursache**
- [x] Ueberlagern sich Pred- und GT-Konturen im Overlay-Plot ueberhaupt raeumlich? (Abschnitt 4) — **ja, aber fragmentiert (mehrere kleine rote Polygone pro gruener GT-Krone)**
- [x] Ist die IoU-Verteilung bimodal (~0) oder verschoben (0.1-0.4)? (Abschnitt 6) — **verschoben: median best-IoU 0.211 bei ueberlappenden Paaren, nur 45/357 GT komplett bei IoU=0**
- [x] Wie hoch ist die mittlere Fragment-Anzahl pro GT-Krone? (Abschnitt 7) — **2.65, 161/357 GT-Kronen fragmentiert (>1 Pred-Polygon)**
- [x] Verbessert groesseres `min_dist`/`sigma` die F1 spuerbar? (Abschnitt 8) — **ja: F1 0.173 -> 0.331 bei min_dist=30, sigma=2; aber fn bleibt fast konstant (~265-269) -> Postprocessing loest kein Recall-Problem**
- [x] Verbessert ein anderer `binary_threshold`/`label_threshold` die F1 zusaetzlich? (Abschnitt 9) — **kaum: bestenfalls F1 0.177 bei bt=0.05, deutlich schlechter bei hoeheren Schwellen. Bestaetigt: nicht der Haupthebel.**
- [x] Sind die 45 komplett verpassten GT-Kronen systematisch kleiner/niedriger oder hat das Netz dort einfach keine Mask-Aktivierung? (Abschnitt 10) — **ja zu beidem: mean_mask_prob median 0.037 (vs. 0.915 bei matched), kleinere Flaeche (9.7 vs. 57.6 m^2), niedrigeres nDOM (0.125 vs. 0.40). Visuelle Stichprobe (5 groesste): 4/5 eindeutig Baeume, 1/5 zu dunkel (vermutlich Schatten) — kein Annotations-Rauschen, echtes Erkennungsdefizit.**
- [ ] Sind kleine/niedrige Kronen in den Trainingsdaten unterrepraesentiert im Vergleich zum Test? (Abschnitt 11)

Bisheriges Bild: zwei getrennte Fehlerquellen. "weak" (237/357, Ueber-Segmentierung) ist ein Postprocessing-Fix (`min_dist=30, sigma=2`, +0.16 F1 auf BotGarten). "missed" (45/357, echte Baeume mit ~0 Mask-Aktivierung) ist ein Modell-/Trainingsdefizit fuer kleine/niedrige Kronen, das kein Postprocessing-Tuning loest. Abschnitt 11 prueft, ob das an fehlenden Trainingsbeispielen dieser Groessenklasse liegt.

Naechste Schritte:
1. `postprocessing.min_dist: 30`, `sigma: 2` in `3_Model/configs/finetune_v1.yaml` uebernehmen, `evaluate.py` auf dem vollen Testsplit (beide Gebiete) laufen lassen, um zu bestaetigen, dass der Gewinn generalisiert.
2. Falls Abschnitt 11 zeigt, dass kleine Kronen im Training unterrepraesentiert sind: `KielPatchDataset` (`3_Model/src/dataset.py`) auf gezieltes Sampling von Patches mit kleinen/niedrigen Kronen umstellen (z.B. Patch-Auswahl nach GT-Kronengroesse gewichten), statt rein zufaellig zu samplen.
3. Falls die Groessenverteilungen aehnlich sind: das Problem liegt eher am Loss (kleine Kronen tragen wenig zum pixelweisen `loss_mask`/`loss_dist` bei) -> instanz- oder groessengewichteten Loss-Term in `train.py::main` in Erwaegung ziehen.

In [ ]:
from rasterio.features import geometry_mask
import pandas as pd

def crown_stats(poly, mask_arr, height_arr, transform):
    inside = geometry_mask([poly], out_shape=mask_arr.shape, transform=transform, invert=True)
    n_px = inside.sum()
    if n_px == 0:
        return np.nan, np.nan, 0
    return float(mask_arr[inside].mean()), float(height_arr[inside].mean()), int(n_px)

height = img[5]  # nDOM channel, normalized [0,1]

rows = []
for g, best_iou in zip(gt_polys, gt_best_iou):
    if best_iou >= 0.5:
        cat = "matched"
    elif best_iou > 0.0:
        cat = "weak"
    else:
        cat = "missed"
    mean_mask, mean_height, n_px = crown_stats(g, mask, height, transform)
    rows.append({"category": cat, "area_m2": g.area, "mean_mask_prob": mean_mask, "mean_height": mean_height, "n_px": n_px})

df = pd.DataFrame(rows)
print(df.groupby("category").size())
print()
print(df.groupby("category")[["area_m2", "mean_mask_prob", "mean_height"]].agg(["median", "mean"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["area_m2", "mean_mask_prob", "mean_height"]):
    df.boxplot(column=col, by="category", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle(f"{AREA} — Merkmale nach Match-Kategorie")
plt.tight_layout()
plt.show()

### Spot-Check: groesste "missed" GT-Kronen visuell pruefen

Damit laesst sich ausschliessen, dass es Annotations-Rauschen ist (z.B. Hecken/Straeucher, die als Baum markiert wurden) statt eines echten Modell-Defizits. Zeigt RGB + nDOM fuer die flaechenmaessig groessten `missed`-Kronen (die kleinsten waeren ohnehin am ehesten Grenzfaelle).

In [ ]:
N_SHOW = 5
PAD_M = 5  # meters of context padding around each crown

missed_idx = df[df["category"] == "missed"].sort_values("area_m2", ascending=False).index[:N_SHOW]
inv = ~transform

fig, axes = plt.subplots(len(missed_idx), 2, figsize=(8, 4 * len(missed_idx)))
if len(missed_idx) == 1:
    axes = axes[None, :]

for row, idx in enumerate(missed_idx):
    poly = gt_polys[idx]
    minx, miny, maxx, maxy = poly.bounds
    minx, miny, maxx, maxy = minx - PAD_M, miny - PAD_M, maxx + PAD_M, maxy + PAD_M

    corners_px = [inv * (x, y) for x, y in [(minx, miny), (minx, maxy), (maxx, miny), (maxx, maxy)]]
    cols = [p[0] for p in corners_px]
    rows_ = [p[1] for p in corners_px]
    c0, c1 = max(int(min(cols)), 0), min(int(max(cols)), rgb.shape[1])
    r0, r1 = max(int(min(rows_)), 0), min(int(max(rows_)), rgb.shape[0])

    coords = [(px - c0, py - r0) for px, py in (inv * (x, y) for x, y in poly.exterior.coords)]

    axes[row, 0].imshow(rgb[r0:r1, c0:c1])
    axes[row, 0].add_patch(MplPolygon(coords, closed=True, fill=False, edgecolor="lime", linewidth=2))
    axes[row, 0].set_title(f"missed idx={idx}  area={df.loc[idx, 'area_m2']:.1f} m^2  RGB")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(height[r0:r1, c0:c1], cmap="viridis", vmin=0, vmax=1)
    axes[row, 1].add_patch(MplPolygon(coords, closed=True, fill=False, edgecolor="red", linewidth=2))
    axes[row, 1].set_title(f"nDOM  mean_mask_prob={df.loc[idx, 'mean_mask_prob']:.3f}")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

## 11. Trainings- vs. Test-Kronengroessenverteilung

Prueft die Hypothese, dass kleine/niedrige Kronen (wie die 45 `missed` aus Abschnitt 10) in den Trainingsdaten unterrepraesentiert sind. `_summer`-Varianten teilen sich die GT-Datei mit ihrem Basis-Gebiet (siehe `finetune_v1.yaml`), werden hier dedupliziert, damit Kronen nicht doppelt gezaehlt werden.

Falls der Anteil kleiner Kronen im Training deutlich niedriger ist als im Test: das Netz hatte schlicht zu wenige Beispiele dieser Groessenklasse -> gezieltes Oversampling/Patch-Sampling auf kleine Kronen ist der naechste Finetuning-Hebel. Falls die Verteilungen aehnlich sind: das Problem liegt eher am Loss/Modell (kleine Kronen tragen wenig zum pixelweisen Loss bei), nicht an fehlenden Beispielen.

In [ ]:
import pandas as pd

train_gt_files = sorted({item["gt_file"] for item in cfg["data"]["train_areas"] if "gt_file" in item})
valid_gt_files = sorted({item["gt_file"] for item in cfg["data"]["valid_areas"] if "gt_file" in item})
test_gt_files  = sorted({item["gt_file"] for item in cfg["data"]["test_areas"]  if "gt_file" in item})
print("train GT files:", train_gt_files)
print("test  GT files:", test_gt_files)

def load_areas_m2(gt_files):
    gdfs = [gpd.read_file(base / f) for f in gt_files]
    gdf_all = pd.concat(gdfs, ignore_index=True)
    gdf_all = gpd.GeoDataFrame(gdf_all, geometry="geometry")
    if gdf_all.crs != raster_crs:
        gdf_all = gdf_all.to_crs(raster_crs)
    return gdf_all.geometry.area

train_area_m2 = load_areas_m2(train_gt_files)
valid_area_m2 = load_areas_m2(valid_gt_files)
test_area_m2  = load_areas_m2(test_gt_files)

print(f"\ntrain: n={len(train_area_m2)}  median={train_area_m2.median():.1f} m^2  mean={train_area_m2.mean():.1f} m^2")
print(f"valid: n={len(valid_area_m2)}  median={valid_area_m2.median():.1f} m^2  mean={valid_area_m2.mean():.1f} m^2")
print(f"test : n={len(test_area_m2)}  median={test_area_m2.median():.1f} m^2  mean={test_area_m2.mean():.1f} m^2")

# Schwelle grob aus Abschnitt 10 (missed: median 9.7 m^2, mean 15.4 m^2)
small_thresh = 16
print(f"\nAnteil Kronen < {small_thresh} m^2 (Groessenklasse der 'missed'-Baeume):")
print(f"  train: {(train_area_m2 < small_thresh).mean():.1%}")
print(f"  valid: {(valid_area_m2 < small_thresh).mean():.1%}")
print(f"  test : {(test_area_m2 < small_thresh).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cap = max(train_area_m2.quantile(0.99), test_area_m2.quantile(0.99))
bins = np.linspace(0, cap, 40)
ax.hist(train_area_m2, bins=bins, alpha=0.5, density=True, label=f"train (n={len(train_area_m2)})")
ax.hist(test_area_m2, bins=bins, alpha=0.5, density=True, label=f"test (n={len(test_area_m2)})")
ax.axvline(small_thresh, color="k", linestyle="--", label=f"'missed'-Groessenklasse ~{small_thresh} m^2")
ax.set_xlabel("Kronenflaeche (m^2)")
ax.set_ylabel("Dichte")
ax.set_title("Kronengroessenverteilung: Train vs. Test")
ax.legend()
plt.show()

## 12. Modell auf 20cm-Sommerbildern (DOP20) auswerten

Wendet das aktuelle Checkpoint auf `stacked_6ch/{area}_summer.tif` an (statt der 7.5cm-Fruehjahrs-TIFs) und matcht gegen dieselben GT-Polygone (Baumpositionen aendern sich saisonal nicht). Braucht `prepare_data.py --summer` vorher fuer die Testgebiete (siehe README-Hinweis von vorhin). Nutzt `min_dist=30, sigma=2` aus Abschnitt 8, weil das auf den 7.5cm-Bildern klar besser war als der Config-Default — gleiche Werte hier als Ausgangspunkt, ggf. fuer 20cm nochmal separat tunen (andere Aufloesung -> andere Watershed-Marker-Dichte).

In [ ]:
pp_tuned = {**pp, "min_dist": 30, "sigma": 2}

summer_rows = []
for item in cfg["data"]["test_areas"]:
    s_area = item["area"]
    summer_path = stacked_dir / f"{s_area}_summer.tif"
    s_gt_shp = base / item["gt_file"]

    if not summer_path.exists():
        print(f"  {s_area}: {summer_path.name} nicht gefunden — erst `prepare_data.py --summer` laufen lassen, skip")
        continue

    with rasterio.open(summer_path) as src:
        img_s = src.read().astype(np.float32)
        transform_s = src.transform
        crs_s = src.crs

    tensor_s = torch.from_numpy(img_s).unsqueeze(0).to(device)
    with torch.no_grad():
        output_s = predict_on_tile(
            model,
            tensor_s,
            patch_size=cfg["data"]["patch_size"],
            local_batch_size=cfg["training"]["batch_size"],
            stride=cfg["data"]["patch_size"] // 2,
        )
    mask_s    = output_s[0, 0].cpu().numpy()
    outline_s = output_s[0, 1].cpu().numpy()
    dist_s    = output_s[0, 2].cpu().numpy()

    pred_polys_s = extract_polygons(mask_s, outline_s, dist_s, transform=transform_s, **pp_tuned)

    gt_gdf_s = gpd.read_file(s_gt_shp)
    if gt_gdf_s.crs != crs_s:
        gt_gdf_s = gt_gdf_s.to_crs(crs_s)
    gt_polys_s = list(gt_gdf_s.geometry)

    tp_s, fp_s, fn_s = match_polygons(pred_polys_s, gt_polys_s, threshold=0.5)
    p_s, r_s, f1_s = compute_metrics(tp_s, fp_s, fn_s)

    summer_rows.append({
        "area": s_area, "pred": len(pred_polys_s), "gt": len(gt_polys_s),
        "tp": tp_s, "fp": fp_s, "fn": fn_s, "precision": p_s, "recall": r_s, "f1": f1_s,
    })
    print(f"  {s_area:20s} pred={len(pred_polys_s):4d}  gt={len(gt_polys_s):4d}  P={p_s:.3f}  R={r_s:.3f}  F1={f1_s:.3f}")

summer_df = pd.DataFrame(summer_rows)
print()
print(summer_df)

if len(summer_df):
    total_tp, total_fp, total_fn = summer_df["tp"].sum(), summer_df["fp"].sum(), summer_df["fn"].sum()
    p_micro, r_micro, f1_micro = compute_metrics(total_tp, total_fp, total_fn)
    print(f"\nMicro-average (summer, 20cm)  P={p_micro:.3f}  R={r_micro:.3f}  F1={f1_micro:.3f}")
    print("Zum Vergleich: spring/7.5cm Micro-average steht in 3_Model/runs/v1/eval_test.csv")

## 13. Modell auf 20cm-Fruehjahrsbildern (aus 7.5cm resampled) auswerten

Abschnitt 12 (DOP20-Sommerbilder) aendert zwei Dinge gleichzeitig: Aufloesung (7.5cm -> 20cm) UND Saison (Fruehjahr -> Sommer). Hier wird stattdessen die vorhandene 7.5cm-Fruehjahrs-TIF direkt im Notebook per bilinearem Resampling auf ~20cm heruntergerechnet — die Saison bleibt Fruehjahr, es aendert sich nur die Aufloesung. Keine zusaetzlichen Dateien noetig (rein on-the-fly, kein `prepare_data.py`-Lauf erforderlich).

Damit laesst sich der Sommer-Einbruch aus Abschnitt 12 zerlegen:
- Faellt die F1 hier aehnlich stark wie in Abschnitt 12 -> die Aufloesung ist der dominante Faktor.
- Faellt sie hier deutlich weniger -> die Saison (Laubzustand, Farbverschiebung, NDVI-Domaenengap) ist der dominante Faktor.

Nutzt dieselben getunten Postprocessing-Parameter (`pp_tuned` aus Abschnitt 8/12).

In [ ]:
from rasterio.enums import Resampling as RS

SPRING20_SCALE = 20.0 / 7.5  # 20cm vs. 7.5cm — isoliert den reinen Aufloesungseffekt (Saison bleibt Fruehjahr)

spring20_rows = []
for item in cfg["data"]["test_areas"]:
    area = item["area"]
    spring_path = stacked_dir / f"{area}.tif"  # bestehende 7.5cm-Fruehjahrs-TIF
    gt_shp = base / item["gt_file"]

    if not spring_path.exists():
        print(f"  {area}: {spring_path.name} nicht gefunden, skip")
        continue

    with rasterio.open(spring_path) as src:
        H20 = max(1, round(src.height / SPRING20_SCALE))
        W20 = max(1, round(src.width / SPRING20_SCALE))
        img20 = src.read(out_shape=(src.count, H20, W20), resampling=RS.bilinear).astype(np.float32)
        transform20 = src.transform * src.transform.scale(src.width / W20, src.height / H20)
        crs20 = src.crs

    tensor20 = torch.from_numpy(img20).unsqueeze(0).to(device)
    with torch.no_grad():
        output20 = predict_on_tile(
            model,
            tensor20,
            patch_size=cfg["data"]["patch_size"],
            local_batch_size=cfg["training"]["batch_size"],
            stride=cfg["data"]["patch_size"] // 2,
        )
    mask20    = output20[0, 0].cpu().numpy()
    outline20 = output20[0, 1].cpu().numpy()
    dist20    = output20[0, 2].cpu().numpy()

    pred_polys20 = extract_polygons(mask20, outline20, dist20, transform=transform20, **pp_tuned)

    gt_gdf20 = gpd.read_file(gt_shp)
    if gt_gdf20.crs != crs20:
        gt_gdf20 = gt_gdf20.to_crs(crs20)
    gt_polys20 = list(gt_gdf20.geometry)

    tp20, fp20, fn20 = match_polygons(pred_polys20, gt_polys20, threshold=0.5)
    p20, r20, f120 = compute_metrics(tp20, fp20, fn20)

    spring20_rows.append({
        "area": area, "pred": len(pred_polys20), "gt": len(gt_polys20),
        "tp": tp20, "fp": fp20, "fn": fn20, "precision": p20, "recall": r20, "f1": f120,
    })
    print(f"  {area:20s} pred={len(pred_polys20):4d}  gt={len(gt_polys20):4d}  P={p20:.3f}  R={r20:.3f}  F1={f120:.3f}")

spring20_df = pd.DataFrame(spring20_rows)
print()
print(spring20_df)

if len(spring20_df):
    total_tp, total_fp, total_fn = spring20_df["tp"].sum(), spring20_df["fp"].sum(), spring20_df["fn"].sum()
    p_micro, r_micro, f1_micro = compute_metrics(total_tp, total_fp, total_fn)
    print(f"\nMicro-average (spring, resampled 20cm)  P={p_micro:.3f}  R={r_micro:.3f}  F1={f1_micro:.3f}")
    print("Vergleich: spring/7.5cm in eval_test.csv, summer/20cm in Abschnitt 12.")
    print("Differenz dieser Zeile zu Abschnitt 12 isoliert den Saisoneffekt; Differenz zu eval_test.csv isoliert den Aufloesungseffekt.")